# Redes Neurais Artificiais 2026.2

- **Disciplina**: Redes Neurais Artificiais 2026.2
- **Professora**: Elloá B. Guedes (ebgcosta@uea.edu.br)  
- **Github**: http://github.com/elloa  
        

Levando em conta a base de dados _Dry Bean Dataset_, esta parte do Projeto Prático diz respeito à proposição e avaliação de múltiplas redes neurais artificiais do tipo feedforward multilayer perceptron para o problema da classificação multiclasse de variedades de feijão seco a partir de características geométricas extraídas de imagens dos grãos.

## Busca em Grade

Uma maneira padrão de escolher os parâmetros de um modelo de Machine Learning é por meio de uma busca em grade via força bruta. O algoritmo da busca em grade é dado como segue:

1. Escolha a métrica de desempenho que você deseja maximizar (por exemplo, o F-Score com média macro, adequado a um problema multiclasse com classes desbalanceadas)  
2. Escolha o algoritmo de Machine Learning (exemplo: redes neurais artificiais). Em seguida, defina os parâmetros ou hiperparâmetros deste tipo de modelo sobre os quais você deseja otimizar (número de épocas, taxa de aprendizado, etc.) e construa um array de valores a serem testados para cada parâmetro ou hiperparâmetro.  
3. Defina a grade de busca, a qual é dada como o produto cartesiano dos valores de cada parâmetro a ser testado. Por exemplo, para os arrays [50, 100, 1000] e [10, 15], tem-se que a grade é [(50,10), (50,15), (100,10), (100,15), (1000,10), (1000,15)].
4. Para cada combinação de parâmetros a serem otimizados, utilize _apenas o conjunto de treinamento_ para realizar uma validação cruzada (holdout ou k-fold) e calcule a métrica de avaliação nos conjuntos de validação gerados por ela. O conjunto de teste não participa da busca. Vamos efetuar validação cruzada holdout 70/30 estratificada.
5. Escolha a combinação de parâmetros que maximiza a métrica de avaliação na validação. Este é o modelo otimizado, que deve então ser reavaliado uma única vez no conjunto de teste.

Por que esta abordagem funciona? Porque a busca em grade efetua uma pesquisa extensiva sobre as possíveis combinações de valores para cada um dos parâmetros a serem ajustados. Para cada combinação, ela estima a performance do modelo em dados não utilizados no seu treinamento (os conjuntos de validação). Por fim, o modelo com melhor métrica de desempenho é escolhido. Tem-se então que este modelo é o que tem maior chance de generalizar bem mediante dados nunca antes vistos, e o conjunto de teste, que não foi utilizado na busca, fornece a estimativa final e não enviesada desse desempenho.

## Efetuando a Busca em Grade sobre Hiperparâmetros das Top-6 RNAs

Considerando a etapa anterior do projeto prático, foram identificadas pelo menos 6 melhores Redes Neurais para o problema da classificação multiclasse das variedades de feijão no conjunto de dados selecionado. Estas redes podem ter sido obtidas pela estratégia própria ou pela regra da Pirâmide Geométrica, e todas utilizam os 16 atributos numéricos como preditores.

A primeira etapa desta segunda parte do projeto consiste em trazer para este notebook estas seis arquiteturas, ressaltando:

1. Número de neurônios ocultos por camada  
2. Função de Ativação  
3. Estratégia de origem da arquitetura (Própria ou Pirâmide Geométrica)  
4. Desempenho médio ± desvio padrão nos testes anteriores  
5. Número de repetições que a equipe conseguiu realizar para verificar os resultados  

Elabore uma busca em grade sobre estas arquiteturas que contemple variações nos hiperparâmetros a seguir, conforme documentação de [MLPClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html)

A. Solver (não usar o LBFGS, pois é mais adequado para datasets pequenos)  
B. Batch Size  
C. Learning Rate Init  
D. Paciência 
E. Épocas  

Nesta busca em grade, contemple a utilização do objeto [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html), com a métrica escolhida no parâmetro `scoring` (por exemplo, `'f1_macro'`) e a validação cruzada realizada apenas com o conjunto de treinamento (já escalonado conforme a estratégia da etapa anterior).

## Validação Cruzada k-fold

Na elaboração da busca em grade, vamos avaliar os modelos propostos segundo uma estratégia de validação cruzada ainda não explorada até o momento: a validação cruzada *k-fold*. Segundo a mesma, o conjunto de dados é particionado em *k* partes (*folds*): a cada iteração, separa-se uma das partes para validação e o modelo é treinado com as *k-1* partes remanescentes. Valores sugestivos de *k* na literatura são *k* = 3, 5 ou 10, pois o custo computacional desta validação dos modelos é alto. A métrica de desempenho é resultante da média dos desempenhos nas *k* iterações. A figura a seguir ilustra a ideia desta avaliação:

<img src="https://ethen8181.github.io/machine-learning/model_selection/img/kfolds.png" width="600">

Considerando a métrica de desempenho F1-Score (com média macro, dado que o problema possui 7 classes), considere a validação cruzada 5-fold estratificada para aferir os resultados da busca em grade anterior, realizada *apenas com o conjunto de treinamento*. O conjunto de teste deve ser utilizado uma única vez, para avaliar a melhor configuração encontrada.

In [6]:
## Reservado para a importação de bibliotecas
import itertools
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

df = pd.read_excel('DryBeanDataset/Dry_Bean_Dataset.xlsx')
x = df.loc[:, df.columns != 'Class']
y = df['Class']
classes = sorted(y.unique())

# As top-6 redes identificadas na Parte 2 do projeto (3 pela estratégia própria e 3 pela regra da pirâmide geométrica), cada uma definida apenas pela sua arquitetura.
top6_arquiteturas = {
    'A9_own': (64,),
    'A10_own': (100,),
    'A8_own': (50,),
    'P_a3_single': (32,),
    'P_a3_split10': (26, 6),
    'P_a3_split9': (23, 9),
}

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.30, random_state=13, stratify=y)


In [7]:
# Grade de hiperparametros a ser testada para cada uma das top-6 arquiteturas
grade_hiperparametros = {
    'solver': ['adam', 'sgd'],
    'batch_size': [32, 200],
    'learning_rate_init': [0.001, 0.01],
    'patience': [10, 20],
    'max_iter': [100, 200],
}
chaves = list(grade_hiperparametros.keys())
combinacoes = list(itertools.product(*grade_hiperparametros.values()))
print('Combinações de hiperparâmetros por arquitetura:', len(combinacoes))
print('Total de combinações (arquitetura x hiperparâmetros):', len(combinacoes) * len(top6_arquiteturas))


Combinações de hiperparâmetros por arquitetura: 32
Total de combinações (arquitetura x hiperparâmetros): 192


In [8]:
# Validação cruzada 5-fold estratificada, usando APENAS o conjunto de treino. Para cada arquitetura e cada combinação de hiperparametros, calculamos o F1-Score (macro) medio nos 5 folds. O conjunto de teste so sera usado uma unica vez, mais adiante.

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
folds = list(kfold.split(x_train, y_train))

linhas_busca = []
for nome_arq, camadas in top6_arquiteturas.items():
    for combo in combinacoes:
        params = dict(zip(chaves, combo))
        f1_folds = []
        for fold_id, (tr_idx, va_idx) in enumerate(folds):
            xt, xv = x_train.iloc[tr_idx], x_train.iloc[va_idx]
            yt, yv = y_train.iloc[tr_idx], y_train.iloc[va_idx]

            escalonador = StandardScaler().fit(xt)
            xt_s = escalonador.transform(xt)
            xv_s = escalonador.transform(xv)

            rede = MLPClassifier(
                hidden_layer_sizes=camadas, activation='relu',
                solver=params['solver'], batch_size=params['batch_size'],
                learning_rate_init=params['learning_rate_init'],
                early_stopping=True, n_iter_no_change=params['patience'],
                max_iter=params['max_iter'], random_state=fold_id,
            )
            rede.fit(xt_s, yt)
            pred = rede.predict(xv_s)
            f1_folds.append(f1_score(yv, pred, average='macro'))

        linha = dict(params)
        linha['arquitetura'] = nome_arq
        linha['camadas'] = camadas
        linha['f1_medio'] = sum(f1_folds) / len(f1_folds)
        linha['folds_f1'] = f1_folds
        linhas_busca.append(linha)

busca_resultado = pd.DataFrame(linhas_busca)
print('Total de combinacoes avaliadas (arquitetura x hiperparametros):', len(busca_resultado))
print('Total de treinamentos realizados (combinacoes x 5 folds):', len(busca_resultado) * 5)


c:\Users\felip\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\felip\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\felip\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\felip\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the opti

Total de combinacoes avaliadas (arquitetura x hiperparametros): 192
Total de treinamentos realizados (combinacoes x 5 folds): 960


In [9]:
# Top-10 combinações segundo o F1-Score medio na validação cruzada
top10 = busca_resultado.sort_values('f1_medio', ascending=False).head(10)
print(top10[['arquitetura', 'camadas', 'solver', 'batch_size', 'learning_rate_init', 'patience', 'max_iter', 'f1_medio']].to_string(index=False))


arquitetura camadas solver  batch_size  learning_rate_init  patience  max_iter  f1_medio
    A10_own  (100,)   adam          32               0.001        20       100  0.940332
    A10_own  (100,)   adam          32               0.001        20       200  0.940332
P_a3_single   (32,)    sgd          32               0.010        20       200  0.939110
P_a3_single   (32,)    sgd          32               0.010        20       100  0.939110
     A8_own   (50,)   adam          32               0.010        20       200  0.938664
     A8_own   (50,)   adam          32               0.010        20       100  0.938664
    A10_own  (100,)   adam         200               0.010        10       200  0.938190
    A10_own  (100,)   adam         200               0.010        10       100  0.938190
     A8_own   (50,)   adam          32               0.001        10       200  0.938113
     A8_own   (50,)   adam          32               0.001        10       100  0.938113


## Identificando a mellhor solução

Como resultado da busca em grade com validação cruzada 5-fold, identifique o modelo otimizado com melhor desempenho para o problema. Apresente claramente este modelo, sua arquitetura (número de camadas ocultas, neurônios por camada e função de ativação), os hiperparâmetros otimizados (solver, batch size, taxa de aprendizado inicial, paciência e épocas) e os resultados (F1-Score macro) para cada um dos folds avaliados, junto com a média ± desvio padrão entre os folds. Em seguida, reavalie este modelo uma única vez no conjunto de teste, apresentando a matriz de confusão, a acurácia e o F1-Score macro. Esta é a melhor solução identificada em decorrência deste projeto.

In [10]:
import numpy as np

# Recuperando a melhor configuração da tabela de resultados
melhor_linha = busca_resultado.loc[busca_resultado['f1_medio'].idxmax()]
print("=== MELHOR CONFIGURAÇÃO ENCONTRADA ===")
print(f"Arquitetura: {melhor_linha['arquitetura']} - Camadas: {melhor_linha['camadas']}")
print(f"Solver: {melhor_linha['solver']}")
print(f"Batch Size: {melhor_linha['batch_size']}")
print(f"Learning Rate Init: {melhor_linha['learning_rate_init']}")
print(f"Paciência: {melhor_linha['patience']}")
print(f"Épocas Max (max_iter): {melhor_linha['max_iter']}")
print(f"F1-Score Médio nos Folds: {melhor_linha['f1_medio']:.4f}")

f1_folds = melhor_linha['folds_f1']
print(f"F1 por fold: {[f'{f:.4f}' for f in f1_folds]}")
print(f"Média ± Desvio Padrão nos Folds: {np.mean(f1_folds):.4f} ± {np.std(f1_folds):.4f}")

# Reavaliando uma única vez no conjunto de teste
scaler_final = StandardScaler().fit(x_train)
x_train_s = scaler_final.transform(x_train)
x_test_s = scaler_final.transform(x_test)

melhor_modelo = MLPClassifier(
    hidden_layer_sizes=melhor_linha['camadas'],
    activation='relu',
    solver=melhor_linha['solver'],
    batch_size=melhor_linha['batch_size'],
    learning_rate_init=melhor_linha['learning_rate_init'],
    early_stopping=True,
    n_iter_no_change=melhor_linha['patience'],
    max_iter=melhor_linha['max_iter'],
    random_state=42
)
melhor_modelo.fit(x_train_s, y_train)
pred_teste = melhor_modelo.predict(x_test_s)

print("\n=== AVALIAÇÃO NO CONJUNTO DE TESTE ===")
print(f"Acurácia: {accuracy_score(y_test, pred_teste):.4f}")
print(f"F1-Score Macro: {f1_score(y_test, pred_teste, average='macro'):.4f}")
print("Matriz de Confusão:")
print(confusion_matrix(y_test, pred_teste))

=== MELHOR CONFIGURAÇÃO ENCONTRADA ===
Arquitetura: A10_own - Camadas: (100,)
Solver: adam
Batch Size: 32
Learning Rate Init: 0.001
Paciência: 20
Épocas Max (max_iter): 100
F1-Score Médio nos Folds: 0.9403
F1 por fold: ['0.9371', '0.9419', '0.9479', '0.9382', '0.9365']
Média ± Desvio Padrão nos Folds: 0.9403 ± 0.0042

=== AVALIAÇÃO NO CONJUNTO DE TESTE ===
Acurácia: 0.9344
F1-Score Macro: 0.9448
Matriz de Confusão:
[[370   0  18   0   2   3   4]
 [  0 156   1   0   0   0   0]
 [ 16   0 462   0   8   1   2]
 [  0   0   0 984   2  20  58]
 [  0   0   2   6 561   0   9]
 [  3   0   0   7   0 586  12]
 [  0   0   2  65  16  11 697]]


## Empacotando a solução

Suponha que você deve entregar este classificador a uma cooperativa ou órgão responsável pelo controle de qualidade e classificação de grãos de feijão seco. Para tanto, você deve fazer uma preparação do mesmo para utilização neste cenário. Uma vez que já identificou os melhores hiperparâmetros, o passo remanescente consiste em treinar o modelo com estes valores e todos os dados disponíveis, salvando o modelo treinado (pesos da rede e o escalonador dos atributos) ao final para entrega ao cliente. Assim, finalize o projeto prático realizando tais passos.

1. Consulte a documentação a seguir:
https://scikit-learn.org/stable/modules/model_persistence.html  
2. Treine o modelo com todos os dados, utilizando a arquitetura e os hiperparâmetros otimizados. O escalonador (`StandardScaler`) deve ser ajustado com todos os dados e salvo junto com a rede, preferencialmente em um único `Pipeline`  
3. Salve o modelo em disco  
4. Construa uma rotina que recupere o modelo em disco  
5. Mostre que a rotina é funcional, fazendo previsões com todos os elementos do dataset e exibindo uma matriz de confusão das mesmas. Como o modelo foi treinado com todos os dados, esta matriz serve apenas para verificar o funcionamento da rotina e não como estimativa de desempenho, que é dada pelos resultados da validação cruzada e do conjunto de teste

In [11]:
pipeline_final = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(
        hidden_layer_sizes=melhor_linha['camadas'],
        activation='relu',
        solver=melhor_linha['solver'],
        batch_size=melhor_linha['batch_size'],
        learning_rate_init=melhor_linha['learning_rate_init'],
        early_stopping=True,
        n_iter_no_change=melhor_linha['patience'],
        max_iter=melhor_linha['max_iter'],
        random_state=42
    ))
])

print("Treinando o modelo com todos os dados disponíveis...")
pipeline_final.fit(x, y)

# 3. Salvando o modelo em disco
nome_arquivo = 'modelo_dry_bean_pipeline.pkl'
joblib.dump(pipeline_final, nome_arquivo)
print(f"Modelo salvo com sucesso no arquivo: '{nome_arquivo}'")

# 4. Rotina para recuperar o modelo do disco
print("Recuperando o modelo salvo em disco...")
modelo_carregado = joblib.load(nome_arquivo)

# 5. Verificando o funcionamento com previsões no dataset completo
pred_completo = modelo_carregado.predict(x)
print("\n=== VALIDAÇÃO DA ROTINA DE CARREGAMENTO (Dataset Completo) ===")
print(f"Acurácia no dataset completo: {accuracy_score(y, pred_completo):.4f}")
print("Matriz de Confusão:")
print(confusion_matrix(y, pred_completo))

Treinando o modelo com todos os dados disponíveis...
Modelo salvo com sucesso no arquivo: 'modelo_dry_bean_pipeline.pkl'
Recuperando o modelo salvo em disco...

=== VALIDAÇÃO DA ROTINA DE CARREGAMENTO (Dataset Completo) ===
Acurácia no dataset completo: 0.9376
Matriz de Confusão:
[[1256    0   36    0    3    8   19]
 [   0  522    0    0    0    0    0]
 [  47    0 1552    0   16    3   12]
 [   0    0    0 3281    4   36  225]
 [   4    0   28   17 1844    0   35]
 [   9    0    1   32    0 1935   50]
 [   6    0    6  199   32   22 2371]]
